# Dual-agent analysis

This notebook reports the non-Hard benchmark (16 domain/scenario combinations), followed by
**the saved historical single-agent vs dual-agent comparison** across all available scenarios.
Dual-agent results are grouped by reasoner/matcher pair, with repeated experiment summaries
averaged within each domain/scenario combination. Each experiment contains five runs.
Source CSVs and the saved backup are unchanged; no new model evaluations are run.
Qwen is displayed as **Qwen3:4b, no thinking**.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.analysis_reporting import (
    OUTPUT, DOMAINS, MODEL_ALIASES, benchmark_frame, scenario_name,
    export_table, bar_chart, save_figure,
)
CHARTS = OUTPUT / "dual_agent_analysis"
CHARTS.mkdir(parents=True, exist_ok=True)
path = Path("eval_results_dual_agent.csv")
source_hash = hashlib.sha256(path.read_bytes()).hexdigest()
raw = pd.read_csv(path)
# Repair the legacy shifted header without changing the source CSV.
assert raw.Value.isna().all() and raw.Summary_Metric.isna().all()
raw["Value"] = pd.to_numeric(raw.Metric_Family, errors="raise")
raw["Summary_Metric"] = raw.Condition
raw["Metric_Family"] = raw.Runs
raw["Runs"] = pd.to_numeric(raw.Baseline_Prompt_Ver, errors="raise")
raw["Condition"] = "Dual_Agent_Store"
raw = raw.drop(columns=["Eval_Prompt_Ver", "Baseline_Prompt_Ver"])
raw["Pair"] = raw.Reasoner_Model.replace(MODEL_ALIASES) + " | " + raw.Matcher_Model.replace(MODEL_ALIASES)
raw["System"] = "Dual-Agent"
keys = ["Timestamp", "Pair", "Domain", "Temp", "Runs", "Metric_Family", "Summary_Metric"]
assert not raw.duplicated(keys).any()
df = benchmark_frame(raw)
assert set(df.Domain) == set(DOMAINS)
assert df.Temp.eq(0).all() and df.Runs.eq(5).all()
# Empty correct/incorrect groups have undefined word means, before pooling experiments.
experiment = ["Timestamp", "Pair", "Domain"]
acc = df[df.Summary_Metric.eq("Average_Accuracy") & df.Metric_Family.eq("End_to_End")]
df = df.merge(acc[experiment + ["Value"]].rename(columns={"Value": "cell_accuracy"}),
              on=experiment, how="left", validate="many_to_one")
undefined_words = ((df.Summary_Metric.eq("TPCA_Words_Correct") & df.cell_accuracy.eq(0)) |
                   (df.Summary_Metric.eq("TPCA_Words_Wrong") & df.cell_accuracy.eq(1)))
export_table(df[undefined_words & df.Value.notna()], "undefined_word_means_audit", CHARTS)
df.loc[undefined_words, "Value"] = np.nan
df = df.drop(columns="cell_accuracy")
export_table(df[df.Summary_Metric.str.contains("Variance|StdDev")], "variability_cells", CHARTS)
# Stored run variances are not averaged; descriptive SD below is across combinations.
df = df[~df.Summary_Metric.str.contains("Variance|StdDev")]
df = df.groupby(["Pair", "System", "Domain", "Metric_Family", "Summary_Metric"], as_index=False).agg(
    Value=("Value", "mean"), experiment_count=("Value", "count"))
df["Scenario"] = df.Domain.map(scenario_name)
export_table(df, "metric_cells", CHARTS)
expected = df[["Pair", "System"]].drop_duplicates().merge(pd.DataFrame({"Domain": DOMAINS}), how="cross")
acc = df[df.Summary_Metric.eq("Average_Accuracy") & df.Metric_Family.eq("End_to_End")]
coverage = expected.merge(acc[["Pair", "System", "Domain", "Value"]],
                          on=["Pair", "System", "Domain"], how="left", validate="one_to_one")
missing = coverage[coverage.Value.isna()]
export_table(missing, "missing_accuracy_combinations", CHARTS)
assert missing.empty and len(coverage) == 4 * 16
assert acc.experiment_count.eq(2).all()
grid = expected.merge(df[["Metric_Family", "Summary_Metric"]].drop_duplicates(), how="cross").merge(
    df, on=["Pair", "System", "Domain", "Metric_Family", "Summary_Metric"],
    how="left", validate="one_to_one")
metric_coverage = grid.groupby(["Pair", "System", "Metric_Family", "Summary_Metric"]).Value.agg(
    available="count", expected="size")
metric_coverage["missing"] = metric_coverage.expected - metric_coverage.available
export_table(metric_coverage, "metric_coverage", CHARTS)
export_table(grid[grid.Value.isna()], "missing_metric_values", CHARTS)
display(metric_coverage[metric_coverage.missing.gt(0)])


,,,,available,expected,missing
Pair,System,Metric_Family,Summary_Metric,,,
gemma3:1b | ministral-3:3b,Dual-Agent,Efficiency,TPCA_Words_Wrong,15,16,1
"ministral-3:3b | Qwen3:4b, no thinking",Dual-Agent,Efficiency,TPCA_Words_Wrong,14,16,2
ministral-3:3b | gemma3:1b,Dual-Agent,Efficiency,TPCA_Words_Wrong,15,16,1


## Binding accuracy vs end-to-end accuracy

**Binding accuracy** evaluates Agent 1 (the reasoner): does its conclusion contain the expected
answer phrase, using a case-insensitive substring check? For each run, it is correct conclusions
/ scored conclusions. A conclusion is scored only when answer options and a nonempty conclusion
exist; a run with none scored is recorded as zero. It does not evaluate Agent 2's final choice.

**End-to-end accuracy** evaluates the complete pipeline: does Agent 2's final option label equal
the gold label? It is correct final answers / all evaluated turns, so failures count as incorrect.
For example, a correct reasoner conclusion followed by a wrong option selection passes binding
but fails end-to-end. Missing conclusions also make the denominators differ, so the gap is not
simply the matcher's error rate. Use **end-to-end accuracy** for comparison with single-agent accuracy.

The CSV's `Average_Accuracy` averages these per-run ratios. The charts below average repeated
experiment summaries within each combination, then weight combinations equally. SD uses ddof=1
across combination accuracies, not per-run uncertainty.


In [2]:
for family in ["Binding", "End_to_End"]:
    cells = df[df.Metric_Family.eq(family) & df.Summary_Metric.eq("Average_Accuracy")].copy()
    for dimension in ["Pair", "Scenario", "Domain"]:
        stats = cells.groupby([dimension, "System"]).Value.agg(mean="mean", sd="std", n="count")
        export_table(stats, family + "_accuracy_by_" + dimension.lower(), CHARTS)
        display(stats)
        bar_chart(stats["mean"].unstack(), family + " accuracy ↑ by " + dimension.lower(), "Accuracy ↑", (0, 1), folder=CHARTS)
    per_pair_scenario = cells.groupby(["Pair", "Scenario", "System"]).Value.agg(mean="mean", sd="std", n="count")
    export_table(per_pair_scenario, family + "_accuracy_pair_scenario", CHARTS)
    display(per_pair_scenario)


,,mean,sd,n
Pair,System,,,
gemma3:1b | gemma3:1b,Dual-Agent,0.61875,0.349130,16
gemma3:1b | ministral-3:3b,Dual-Agent,0.62625,0.346119,16
"ministral-3:3b | Qwen3:4b, no thinking",Dual-Agent,0.78750,0.152206,16
ministral-3:3b | gemma3:1b,Dual-Agent,0.79000,0.156589,16


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,,mean,sd,n
Scenario,System,,,
absurd,Dual-Agent,0.82875,0.111527,16
absurd_temporal,Dual-Agent,0.82875,0.160286,16
belief_maintenance,Dual-Agent,0.80500,0.078825,16
grounding,Dual-Agent,0.36000,0.322883,16


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,,mean,sd,n
Domain,System,,,
alien_clinic_absurd,Dual-Agent,0.7575,0.112657,4
alien_clinic_absurd_temporal,Dual-Agent,0.5700,0.034641,4
alien_clinic_belief_maintenance,Dual-Agent,0.8775,0.026300,4
alien_clinic_grounding,Dual-Agent,0.4775,0.424608,4
crime_scene_absurd,Dual-Agent,0.8850,0.010000,4
crime_scene_absurd_temporal,Dual-Agent,0.8950,0.010000,4
crime_scene_belief_maintenance,Dual-Agent,0.8000,0.000000,4
crime_scene_grounding,Dual-Agent,0.2225,0.158824,4
loan_absurd,Dual-Agent,0.8175,0.096047,4


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


... (4 outputs left out)

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,,mean,sd,n
Domain,System,,,
alien_clinic_absurd,Dual-Agent,0.7550,0.277429,4
alien_clinic_absurd_temporal,Dual-Agent,0.5875,0.032016,4
alien_clinic_belief_maintenance,Dual-Agent,0.8775,0.121209,4
alien_clinic_grounding,Dual-Agent,0.6750,0.270000,4
crime_scene_absurd,Dual-Agent,0.9625,0.062383,4
crime_scene_absurd_temporal,Dual-Agent,0.8650,0.070000,4
crime_scene_belief_maintenance,Dual-Agent,0.8750,0.088129,4
crime_scene_grounding,Dual-Agent,0.4975,0.227797,4
loan_absurd,Dual-Agent,0.8375,0.137689,4


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean  \
Pair                                   Scenario           System               
gemma3:1b | gemma3:1b                  absurd             Dual-Agent  0.7100   
                                       absurd_temporal    Dual-Agent  0.7025   
                                       belief_maintenance Dual-Agent  0.6875   
                                       grounding          Dual-Agent  0.1975   
gemma3:1b | ministral-3:3b             absurd             Dual-Agent  0.8675   
                                       absurd_temporal    Dual-Agent  0.8225   
                                       belief_maintenance Dual-Agent  0.7775   
                                       grounding          Dual-Agent  0.3675   
ministral-3:3b | Qwen3:4b, no thinking absurd             Dual-Agent  0.9350   
                                       absurd_temporal    Dual-Agent  0.8400   
                                       belief_maintenance Dual-Agent  0.8950   
                                       grounding          Dual-Agent  0.8100   
ministral-3:3b | gemma3:1b             absurd             Dual-Agent  0.9275   
                                       absurd_temporal    Dual-Agent  0.8125   
                                       belief_maintenance Dual-Agent  0.8975   
                                       grounding          Dual-Agent  0.8250   

                                                                            sd  \
Pair                                   Scenario           System                 
gemma3:1b | gemma3:1b                  absurd             Dual-Agent  0.243173   
                                       absurd_temporal    Dual-Agent  0.117296   
                                       belief_maintenance Dual-Agent  0.094296   
                                       grounding          Dual-Agent  0.114419   
gemma3:1b | ministral-3:3b             absurd             Dual-Agent  0.141745   
                                       absurd_temporal    Dual-Agent  0.193972   
                                       belief_maintenance Dual-Agent  0.103401   
                                       grounding          Dual-Agent  0.191377   
ministral-3:3b | Qwen3:4b, no thinking absurd             Dual-Agent  0.085440   
                                       absurd_temporal    Dual-Agent  0.162481   
                                       belief_maintenance Dual-Agent  0.094692   
                                       grounding          Dual-Agent  0.118322   
ministral-3:3b | gemma3:1b             absurd             Dual-Agent  0.084607   
                                       absurd_temporal    Dual-Agent  0.143614   
                                       belief_maintenance Dual-Agent  0.069462   
                                       grounding          Dual-Agent  0.110905   

                                                                      n  
Pair                                   Scenario           System         
gemma3:1b | gemma3:1b                  absurd             Dual-Agent  4  
                                       absurd_temporal    Dual-Agent  4  
                                       belief_maintenance Dual-Agent  4  
                                       grounding          Dual-Agent  4  
gemma3:1b | ministral-3:3b             absurd             Dual-Agent  4  
                                       absurd_temporal    Dual-Agent  4  
                                       belief_maintenance Dual-Agent  4  
                                       grounding          Dual-Agent  4  
ministral-3:3b | Qwen3:4b, no thinking absurd             Dual-Agent  4  
                                       absurd_temporal    Dual-Agent  4  
                                       belief_maintenance Dual-Agent  4  
                                       grounding          Dual-Agent  4  
ministral-3:3b | gemma3:1b             absurd             Dual-Agent  4  
                                       absur

## Calibration, stability, evidence and response length

Means and SDs use combination summaries after averaging repeated experiments. Available combination counts are shown; missing values are omitted. Undefined word means are audited before aggregation.

In [3]:
specs = [("Matcher", "Brier_Score", "Brier ↓"), ("Matcher", "ECE", "ECE ↓"),
         ("Matcher", "MacroCE", "MacroCE ↓"), ("Stability", "AFR_Answer_Flip_Rate", "Answer flip rate ↓"),
         ("Stability", "EFR_Extraction_Failure_Rate", "Extraction failure ↓"),
         ("Reasoning", "Average_Evidence_F1", "Evidence F1 ↑")]
for family, metric, label in specs:
    cells = df[df.Metric_Family.eq(family) & df.Summary_Metric.eq(metric)]
    if cells.empty:
        print("Unavailable:", family, metric)
        continue
    stats = cells.groupby(["Pair", "System"]).Value.agg(mean="mean", sd="std", n="count")
    export_table(stats, metric + "_summary", CHARTS)
    display(stats)
    bar_chart(stats["mean"].unstack(), label + " by pair · available support", label, folder=CHARTS)
summary = df.groupby(["Pair", "System", "Metric_Family", "Summary_Metric"]).Value.agg(mean="mean", sd="std", n="count")
export_table(summary, "metric_summaries", CHARTS)
display(summary)


,,mean,sd,n
Pair,System,,,
gemma3:1b | gemma3:1b,Dual-Agent,0.426600,0.263237,16
gemma3:1b | ministral-3:3b,Dual-Agent,0.249375,0.201963,16
"ministral-3:3b | Qwen3:4b, no thinking",Dual-Agent,0.126234,0.118328,16
ministral-3:3b | gemma3:1b,Dual-Agent,0.134869,0.107300,16


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,,mean,sd,n
Pair,System,,,
gemma3:1b | gemma3:1b,Dual-Agent,0.427950,0.262369,16
gemma3:1b | ministral-3:3b,Dual-Agent,0.247100,0.219355,16
"ministral-3:3b | Qwen3:4b, no thinking",Dual-Agent,0.126081,0.116819,16
ministral-3:3b | gemma3:1b,Dual-Agent,0.136553,0.109583,16


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,,mean,sd,n
Pair,System,,,
gemma3:1b | gemma3:1b,Dual-Agent,0.503675,0.008466,16
gemma3:1b | ministral-3:3b,Dual-Agent,0.447588,0.127589,16
"ministral-3:3b | Qwen3:4b, no thinking",Dual-Agent,0.404975,0.201172,16
ministral-3:3b | gemma3:1b,Dual-Agent,0.454381,0.136574,16


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


... (3 outputs left out)

/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,,mean,sd,n
Pair,System,,,
gemma3:1b | gemma3:1b,Dual-Agent,0.445206,0.168296,16
gemma3:1b | ministral-3:3b,Dual-Agent,0.452072,0.169133,16
"ministral-3:3b | Qwen3:4b, no thinking",Dual-Agent,0.316481,0.179629,16
ministral-3:3b | gemma3:1b,Dual-Agent,0.316119,0.182781,16


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


mean  \
Pair                       System     Metric_Family Summary_Metric                              
gemma3:1b | gemma3:1b      Dual-Agent Binding       Average_Accuracy                 0.618750   
                                                    Standard_Deviation_Accuracy      0.029066   
                                      Efficiency    TPCA_Words_Correct              60.721875   
                                                    TPCA_Words_Wrong                61.606250   
                                                    WCT_WallClock_PerTurn_Seconds   12.162428   
...                                                                                       ...   
ministral-3:3b | gemma3:1b Dual-Agent Retrieval     Average_BCR_BeliefCoverage       1.000000   
                                                    Average_SBIR_SpuriousInjection   0.237544   
                                      Stability     AFR_Answer_Flip_Rate             0.030000   
                                                    DS_Determinism_Score             0.906250   
                                                    EFR_Extraction_Failure_Rate      0.000000   

                                                                                           sd  \
Pair                       System     Metric_Family Summary_Metric                              
gemma3:1b | gemma3:1b      Dual-Agent Binding       Average_Accuracy                 0.349130   
                                                    Standard_Deviation_Accuracy      0.025667   
                                      Efficiency    TPCA_Words_Correct              11.307460   
                                                    TPCA_Words_Wrong                 7.390690   
                                                    WCT_WallClock_PerTurn_Seconds    4.483307   
...                                                                                       ...   
ministral-3:3b | gemma3:1b Dual-Agent Retrieval     Average_BCR_BeliefCoverage       0.000000   
                                                    Average_SBIR_SpuriousInjection   0.076739   
                                      Stability     AFR_Answer_Flip_Rate             0.032249   
                                                    DS_Determinism_Score             0.092871   
                                                    EFR_Extraction_Failure_Rate      0.000000   

                                                                                     n  
Pair                       System     Metric_Family Summary_Metric                      
gemma3:1b | gemma3:1b      Dual-Agent Binding       Average_Accuracy                16  
                                                    Standard_Deviation_Accuracy     16  
                                      Efficiency    TPCA_Words_Correct              16  
                                                    TPCA_Words_Wrong                16  
                                                    WCT_WallClock_PerTurn_Seconds   16  
...                                                                                 ..  
ministral-3:3b | gemma3:1b Dual-Agent Retrieval     Average_BCR_BeliefCoverage      16  
                                                    Average_SBIR_SpuriousInjection  16  
                                      Stability     AFR_Answer_Flip_Rate            16  
                                                    DS_Determinism_Score            16  
                                                    EFR_Extraction_Failure_Rate     16  

[100 rows x 3 columns]

## Single-agent vs dual-agent — restored backup comparison

This reproduces the backup's **Average Accuracy: Single vs Dual (All Models & Scenarios)** chart:
**57.37% single-agent vs 75.89% dual-agent** (about 57% vs 75.8% in the saved comparison).
These are means of stored summary rows, using all models and scenarios in each original CSV.
Single-agent accuracy pools With Store and No Store, including Hard; dual-agent uses End-to-End
accuracy, including noise scenarios. Thus this is the historical descriptive comparison, not a
matched comparison on the 16-combination benchmark above.

The additional bars use the same historical sources and available values, with counts in the table.
**Evidence F1** measures overlap between cited and expected evidence keys (harmonic mean of evidence
precision and recall); it is not answer-classification F1. Single-agent evidence scores are available
only for With Store. **Brier** measures squared confidence error, **ECE** measures binned confidence–accuracy
gaps, and **MacroCE** balances confidence error across correct and incorrect predictions.
Lower is better for all three calibration metrics; higher is better for accuracy and Evidence F1.
Dual-agent calibration uses the matcher's confidence against final-answer correctness.
These bars average stored calibration summaries, rather than recomputing pooled calibration from
individual predictions; missing values are omitted, never replaced with zero.


In [4]:
single_path = Path("eval_results_normalized.csv")
single_hash = hashlib.sha256(single_path.read_bytes()).hexdigest()
single = pd.read_csv(single_path)
single["Value"] = pd.to_numeric(single.Value, errors="raise")
comparison_specs = [
    ("Accuracy", "Average_Accuracy", "End_to_End", "Average_Accuracy"),
    ("Evidence F1", "Average_Evidence_F1", "Reasoning", "Average_Evidence_F1"),
    ("Brier", "Summary_Metric_Brier_Score", "Matcher", "Brier_Score"),
    ("ECE", "Summary_Metric_ECE", "Matcher", "ECE"),
    ("MacroCE", "Summary_Metric_MacroCE", "Matcher", "MacroCE"),
]
rows = []
for label, single_metric, family, dual_metric in comparison_specs:
    values_by_setup = {
        "Single-Agent": single.loc[single.Summary_Metric.eq(single_metric), "Value"],
        "Dual-Agent": raw.loc[raw.Metric_Family.eq(family) & raw.Summary_Metric.eq(dual_metric), "Value"],
    }
    for setup, values in values_by_setup.items():
        assert values.notna().any(), (label, setup)
        assert values.dropna().between(0, 1).all(), (label, setup)
        rows.append({"Metric": label, "Setup": setup, "Mean": values.mean(),
                     "Available": values.count(), "Missing": values.isna().sum()})
comparison = pd.DataFrame(rows)
export_table(comparison, "historical_single_vs_dual_metrics", CHARTS)
display(comparison)
compare_df = comparison[comparison.Metric.eq("Accuracy")].set_index("Setup")[["Mean"]].rename(
    columns={"Mean": "Average_Accuracy"})
# Regression check against the stored backup output, not hard-coded chart heights.
np.testing.assert_allclose(compare_df.Average_Accuracy, [0.573723, 0.758889], atol=0.0000005, rtol=0)
ax = compare_df.plot(kind="bar", legend=False, figsize=(6, 4))
ax.set_title("Average Accuracy: Single vs Dual (All Models & Scenarios)")
ax.set_ylabel("Average Accuracy")
ax.set_xlabel("Setup")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelrotation=0, pad=8)
ax.bar_label(ax.containers[0], labels=[f"{v:.2%}" for v in compare_df.Average_Accuracy], padding=3)
np.testing.assert_allclose([p.get_height() for p in ax.patches], compare_df.Average_Accuracy)
plt.tight_layout()
scope = "Historical all-scenario sources · different cohorts · available summary means"
export_table(compare_df, "historical_single_vs_dual_accuracy", CHARTS)
save_figure(ax.figure, "historical_single_vs_dual_accuracy", CHARTS, coverage=scope)
plt.show()
plt.close(ax.figure)
for labels, title, ylabel in [
    (["Evidence F1"], "Evidence F1: Single vs Dual ↑", "Evidence F1 ↑"),
    (["Brier", "ECE", "MacroCE"], "Calibration: Single vs Dual ↓", "Calibration error ↓"),
]:
    table = comparison[comparison.Metric.isin(labels)].pivot(index="Metric", columns="Setup", values="Mean")
    table = table.reindex(index=labels, columns=["Single-Agent", "Dual-Agent"])
    bar_chart(table, title, ylabel, (0, 1), folder=CHARTS, coverage=scope)
assert hashlib.sha256(single_path.read_bytes()).hexdigest() == single_hash


,Metric,Setup,Mean,Available,Missing
0,Accuracy,Single-Agent,0.573723,368,0
1,Accuracy,Dual-Agent,0.758889,144,0
2,Evidence F1,Single-Agent,0.296077,184,0
3,Evidence F1,Dual-Agent,0.360324,144,0
4,Brier,Single-Agent,0.305492,364,0
5,Brier,Dual-Agent,0.231301,144,0
6,ECE,Single-Agent,0.313739,364,0
7,ECE,Dual-Agent,0.231165,144,0
8,MacroCE,Single-Agent,0.424168,364,0
9,MacroCE,Dual-Agent,0.458775,144,0


/var/folders/87/1jg3_rzd2dn_ndzg424yjsth0000gn/T/ipykernel_16817/1460658878.py:42: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Single-agent vs dual-agent — Hard and noise excluded

Both sources are restricted to the same 16 domain/scenario combinations: Hard and noise
scenarios are excluded from every metric in this comparison. Accuracy, Evidence F1,
Brier, ECE, and MacroCE retain the historical available-summary averaging and single-agent
condition pooling. Counts are shown below. Single-agent Evidence F1 is available only
for With Store. Model sets and experimental settings still differ between the systems.

The all-scenario backup reproduction above is retained only as a historical reference.


In [5]:
# Apply the shared scenario filter to both systems before selecting any metric.
single_non_hard = benchmark_frame(single)
dual_non_hard = benchmark_frame(raw)
assert set(single_non_hard.Domain) == set(dual_non_hard.Domain) == set(DOMAINS)
assert not single_non_hard.Domain.str.contains("hard|noise").any()
assert not dual_non_hard.Domain.str.contains("hard|noise").any()
assert len(dual_non_hard) < len(raw), "Noise rows must be excluded"
rows = []
for label, single_metric, family, dual_metric in comparison_specs:
    values_by_setup = {
        "Single-Agent": single_non_hard.loc[single_non_hard.Summary_Metric.eq(single_metric), "Value"],
        "Dual-Agent": dual_non_hard.loc[dual_non_hard.Metric_Family.eq(family) & dual_non_hard.Summary_Metric.eq(dual_metric), "Value"],
    }
    for setup, values in values_by_setup.items():
        assert values.notna().any(), (label, setup)
        assert values.dropna().between(0, 1).all(), (label, setup)
        rows.append({"Metric": label, "Setup": setup, "Mean": values.mean(),
                     "Available": values.count(), "Missing": values.isna().sum()})
comparison = pd.DataFrame(rows)
export_table(comparison, "non_hard_single_vs_dual_metrics", CHARTS)
display(comparison)
compare_df = comparison[comparison.Metric.eq("Accuracy")].set_index("Setup")[["Mean"]].rename(
    columns={"Mean": "Average_Accuracy"})
ax = compare_df.plot(kind="bar", legend=False, figsize=(6, 4))
ax.set_title("Average Accuracy: Single vs Dual (Hard & Noise Excluded)")
ax.set_ylabel("Average Accuracy")
ax.set_xlabel("Setup")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelrotation=0, pad=8)
ax.bar_label(ax.containers[0], labels=[f"{v:.2%}" for v in compare_df.Average_Accuracy], padding=3)
np.testing.assert_allclose([p.get_height() for p in ax.patches], compare_df.Average_Accuracy)
plt.tight_layout()
scope = "Hard and noise excluded · same 16 scenario combinations · historical summary means"
export_table(compare_df, "non_hard_single_vs_dual_accuracy", CHARTS)
save_figure(ax.figure, "non_hard_single_vs_dual_accuracy", CHARTS, coverage=scope)
plt.show()
plt.close(ax.figure)
for labels, title, ylabel in [
    (["Evidence F1"], "Evidence F1: Single vs Dual (Hard & Noise Excluded) ↑", "Evidence F1 ↑"),
    (["Brier", "ECE", "MacroCE"], "Calibration: Single vs Dual (Hard & Noise Excluded) ↓", "Calibration error ↓"),
]:
    table = comparison[comparison.Metric.isin(labels)].pivot(index="Metric", columns="Setup", values="Mean")
    table = table.reindex(index=labels, columns=["Single-Agent", "Dual-Agent"])
    bar_chart(table, title, ylabel, (0, 1), folder=CHARTS, coverage=scope,
              name="evidence_f1_single_vs_dual_hard_excluded" if labels == ["Evidence F1"] else "calibration_single_vs_dual_hard_excluded")
assert hashlib.sha256(single_path.read_bytes()).hexdigest() == single_hash

assert hashlib.sha256(path.read_bytes()).hexdigest() == source_hash


,Metric,Setup,Mean,Available,Missing
0,Accuracy,Single-Agent,0.589406,320,0
1,Accuracy,Dual-Agent,0.754687,128,0
2,Evidence F1,Single-Agent,0.288788,160,0
3,Evidence F1,Dual-Agent,0.382470,128,0
4,Brier,Single-Agent,0.278396,316,0
5,Brier,Dual-Agent,0.234270,128,0
6,ECE,Single-Agent,0.286745,316,0
7,ECE,Dual-Agent,0.234421,128,0
8,MacroCE,Single-Agent,0.414617,316,0
9,MacroCE,Dual-Agent,0.452655,128,0


/var/folders/87/1jg3_rzd2dn_ndzg424yjsth0000gn/T/ipykernel_16817/2225669352.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


/Users/mws/Documents/GitHub/Belief-Aware-LLMs/scripts/analysis_reporting.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## Source provenance and checks

In [6]:
assert hashlib.sha256(path.read_bytes()).hexdigest() == source_hash
(CHARTS / "definitions.json").write_text(json.dumps({
    "source_file": str(path), "sha256": source_hash,
    "combinations": list(DOMAINS), "condition": "Dual_Agent_Store", "runs": 5,
    "aggregation": "Equal combination means after averaging repeated experiments within each pair/combination. SD ddof=1 across combinations.",
    "historical_comparison": {"single_file": str(single_path), "sha256": single_hash,
        "scope": "All available historical summary rows, including single-agent Hard and dual-agent noise; single-agent conditions pooled."},
    "non_hard_comparison": "Both systems restricted to the same 16 domain/scenario combinations; Hard and noise excluded.",
    "missingness": "Available values only; no zero imputation.",
    "limitations": "Historical summaries only; different cohorts, no verified question pairing or pooled prediction-level calibration."
}, indent=2) + "\n")
print("PASS: 4 pairs × 16 combinations, repeated experiment coverage, uniqueness, backup accuracy reproduction, source integrity and chart/export agreement.")


PASS: 4 pairs × 16 combinations, repeated experiment coverage, uniqueness, backup accuracy reproduction, source integrity and chart/export agreement.
